<a href="https://colab.research.google.com/github/qmohsu/ENG3004-Economic-Dimension/blob/main/Break_even_analysis_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improved Break-Even Analysis: Aircraft Fleet Modernization

This notebook implements a professional-grade capital budgeting model for a 50-aircraft fleet acquisition. It addresses all 10 deficiencies identified in the [Financial Methodology Audit Report](Financial_Methodology_Audit_Report.md):

| # | Improvement | Source |
|---|---|---|
| 1 | WACC-derived discount rate | [Damodaran](https://pages.stern.nyu.edu/~adamodar/pdfiles/papers/costofcapital.pdf) |
| 2 | CAPM-based cost of equity | [Investopedia](https://www.investopedia.com/terms/c/capm.asp) |
| 3 | MACRS depreciation tax shield | [NBAA](https://nbaa.org/flight-department-administration/tax-issues/depreciation/depreciation-schedule-for-business-aircraft/) |
| 4 | Residual / salvage value | [IATA](https://www.iata.org/contentassets/4a4b100c43794398baf73dcea6b5ad42/airline-disclosure-guide-aircraft-acquisition.pdf) |
| 5 | FCFF-based cash flows | [CFI](https://corporatefinanceinstitute.com/resources/financial-modeling/free-cash-flow-to-firm-fcff) |
| 6 | NPV, IRR, MIRR, PI metrics | [CFA Institute](https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2025/capital-investments-and-capital-allocation) |
| 7 | Bathtub maintenance cost model | [IATA](https://www.iata.org/contentassets/bf8ca67c8bcd4358b3d004b0d6d0916f/mcaa-1sted-2018.pdf) |
| 8 | Revenue growth model | [IATA](https://www.iata.org/en/iata-repository/publications/economic-reports/) |
| 9 | Sensitivity + Monte Carlo analysis | [Investopedia](https://www.investopedia.com/terms/s/sensitivityanalysis.asp) |
| 10 | Financing structure (purchase vs lease) | [IATA](https://www.iata.org/en/publications/manuals/international-aircraft-financing) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

np.random.seed(42)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

---
## 1. Project Parameters

All monetary values in **millions of US dollars (M USD)**.

In [ ]:
NUM_AIRCRAFT       = 50
COST_PER_AIRCRAFT  = 120.0        # $M per aircraft
C0                 = NUM_AIRCRAFT * COST_PER_AIRCRAFT   # Total initial investment: $6,000M
PROJECT_LIFE       = 25           # years

ANNUAL_REVENUE_0   = 15.0         # $M revenue per aircraft in Year 0
FUEL_SAVINGS_PCT   = 0.20         # 20% fuel savings on revenue
MAINT_SAVINGS_BASE = 2.0          # $M maintenance savings per aircraft in Year 1

REVENUE_GROWTH     = 0.03         # 3% nominal annual revenue growth
INFLATION          = 0.025        # 2.5% general inflation
SALVAGE_PCT        = 0.15         # aircraft retain 15% of purchase price at end of life

TAX_RATE           = 0.21         # US federal corporate tax rate
NWC_PCT            = 0.005        # net working capital as % of incremental revenue

print(f"Total initial investment C0 = ${C0:,.0f}M")
print(f"Project horizon            = {PROJECT_LIFE} years")
print(f"Salvage value at year {PROJECT_LIFE}   = ${NUM_AIRCRAFT * COST_PER_AIRCRAFT * SALVAGE_PCT:,.0f}M")

---
## 2. CAPM → Cost of Equity → WACC (Issues 1 & 2)

**CAPM:**  $R_e = R_f + \beta (R_m - R_f)$

**WACC:**  $\text{WACC} = \frac{E}{V} R_e + \frac{D}{V} R_d (1 - T_c)$

Sources: [Damodaran](https://pages.stern.nyu.edu/~adamodar/pdfiles/papers/costofcapital.pdf), [Investopedia CAPM](https://www.investopedia.com/terms/c/capm.asp)

In [ ]:
R_F            = 0.0425    # 10-year US Treasury yield
BETA           = 1.30      # airline equity beta (Damodaran sector data)
EQUITY_PREMIUM = 0.055     # historical equity risk premium
R_D            = 0.045     # pre-tax cost of secured aircraft debt
DEBT_RATIO     = 0.60      # D/V — airlines are typically 50-70% debt-financed
EQUITY_RATIO   = 1.0 - DEBT_RATIO

R_E  = R_F + BETA * EQUITY_PREMIUM
WACC = EQUITY_RATIO * R_E + DEBT_RATIO * R_D * (1 - TAX_RATE)

print("=== CAPM & WACC Derivation ===")
print(f"Risk-free rate  Rf       = {R_F:.2%}")
print(f"Equity beta     β        = {BETA:.2f}")
print(f"Equity premium  (Rm-Rf)  = {EQUITY_PREMIUM:.2%}")
print(f"Cost of equity  Re       = {R_E:.2%}")
print(f"Cost of debt    Rd       = {R_D:.2%}")
print(f"Debt ratio      D/V      = {DEBT_RATIO:.0%}")
print(f"Tax rate        Tc       = {TAX_RATE:.0%}")
print(f"───────────────────────────────")
print(f"WACC                     = {WACC:.4%}")
print(f"\n(Compare: naive model used a flat 8.00% — the derived WACC is {WACC:.2%})")

---
## 3. MACRS Depreciation Tax Shield (Issue 3)

Commercial aircraft (Part 135) use a **7-year MACRS** recovery schedule.

Source: [NBAA](https://nbaa.org/flight-department-administration/tax-issues/depreciation/depreciation-schedule-for-business-aircraft/), [IRS Pub 946](https://www.irs.gov/pub/irs-pdf/p946.pdf)

In [ ]:
MACRS_7YR = np.array([14.29, 24.49, 17.49, 12.49, 8.93, 8.92, 8.93, 4.46]) / 100.0

depreciation = np.zeros(PROJECT_LIFE)
for i, rate in enumerate(MACRS_7YR):
    depreciation[i] = C0 * rate

tax_shield = depreciation * TAX_RATE

print("=== MACRS Depreciation Schedule (fleet-level, $M) ===")
print(f"{'Year':>5} {'MACRS %':>8} {'Depreciation':>14} {'Tax Shield':>12}")
print("─" * 42)
for t in range(PROJECT_LIFE):
    if depreciation[t] > 0:
        print(f"{t+1:5d} {depreciation[t]/C0:8.2%} {depreciation[t]:14,.1f} {tax_shield[t]:12,.1f}")
print(f"{'Total':>5} {'':>8} {depreciation.sum():14,.1f} {tax_shield.sum():12,.1f}")

---
## 4. Bathtub Maintenance Cost Model (Issue 7)

Piecewise escalation replacing the naive constant 5% growth, plus discrete C-check / D-check overlays.

$$C_{\text{maint}}(t) = \begin{cases} C_{\text{base}} & 1 \le t \le 5 \\ C_{\text{base}} (1+g_1)^{t-5} & 6 \le t \le 12 \\ C_{\text{base}} (1+g_1)^7 (1+g_2)^{t-12} & t > 12 \end{cases}$$

Source: [IATA — Managing and Costing for Aging Aircraft](https://www.iata.org/contentassets/bf8ca67c8bcd4358b3d004b0d6d0916f/mcaa-1sted-2018.pdf)

In [ ]:
G1_MID       = 0.03    # mid-life escalation (3%/yr)
G2_LATE      = 0.09    # late-life escalation (9%/yr)
C_CHECK_COST = 2.0     # $M per aircraft per C-check
D_CHECK_COST = 8.0     # $M per aircraft per D-check
C_CHECK_INTERVAL = 7   # years
D_CHECK_INTERVAL = 12  # years

def maintenance_savings_per_aircraft(t):
    """Bathtub-curve maintenance savings per aircraft in year t (1-indexed)."""
    base = MAINT_SAVINGS_BASE
    if t <= 5:
        return base
    elif t <= 12:
        return base * (1 + G1_MID) ** (t - 5)
    else:
        return base * (1 + G1_MID) ** 7 * (1 + G2_LATE) ** (t - 12)

def major_check_capex_per_aircraft(t):
    """Discrete heavy-maintenance CapEx overlays per aircraft."""
    capex = 0.0
    if t > 1 and t % C_CHECK_INTERVAL == 0:
        capex += C_CHECK_COST
    if t > 1 and t % D_CHECK_INTERVAL == 0:
        capex += D_CHECK_COST
    return capex

years = np.arange(1, PROJECT_LIFE + 1)
maint_savings = np.array([maintenance_savings_per_aircraft(t) for t in years])
check_capex   = np.array([major_check_capex_per_aircraft(t) for t in years])

fig, ax1 = plt.subplots()
ax1.plot(years, maint_savings, 'b-o', markersize=4, label='Maint. savings / aircraft')
ax1.set_xlabel('Year')
ax1.set_ylabel('Savings ($M / aircraft)', color='b')
ax2 = ax1.twinx()
ax2.bar(years, check_capex, color='red', alpha=0.4, label='C/D-check CapEx')
ax2.set_ylabel('Check CapEx ($M / aircraft)', color='r')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Bathtub Maintenance Model: Savings & Heavy-Check CapEx')
plt.tight_layout()
plt.show()

---
## 5. Revenue Growth Model (Issue 8)

$R_t = R_0 \cdot (1 + g_R)^t$  where $g_R = 3\%$ nominal.

Fuel savings are computed as a percentage of *growing* revenue, not a static value.

Source: [IATA Economic Reports](https://www.iata.org/en/iata-repository/publications/economic-reports/)

In [ ]:
revenue_per_ac = np.array([ANNUAL_REVENUE_0 * (1 + REVENUE_GROWTH) ** t for t in years])
fuel_savings_per_ac = FUEL_SAVINGS_PCT * revenue_per_ac

print("=== Revenue & Fuel Savings per Aircraft ($M) ===")
print(f"{'Year':>5} {'Revenue':>10} {'Fuel Savings':>14}")
print("─" * 32)
for t in [1, 5, 10, 15, 20, 25]:
    idx = t - 1
    print(f"{t:5d} {revenue_per_ac[idx]:10.2f} {fuel_savings_per_ac[idx]:14.2f}")

---
## 6. Free Cash Flow to Firm — FCFF (Issues 4 & 5)

$$\text{FCFF}_t = (\text{FuelSavings}_t + \text{MaintSavings}_t) \times (1 - T_c) + D_t \times T_c - \text{CheckCapEx}_t - \Delta\text{NWC}_t$$

Terminal cash flow in the final year includes the **salvage value** of the fleet (net of tax on any gain).

Sources: [Investopedia FCFF](https://www.investopedia.com/terms/f/freecashflowfirm.asp), [IATA Disclosure Guide](https://www.iata.org/contentassets/4a4b100c43794398baf73dcea6b5ad42/airline-disclosure-guide-aircraft-acquisition.pdf)

In [ ]:
fleet_fuel_savings  = NUM_AIRCRAFT * fuel_savings_per_ac
fleet_maint_savings = NUM_AIRCRAFT * maint_savings
fleet_check_capex   = NUM_AIRCRAFT * check_capex

gross_savings    = fleet_fuel_savings + fleet_maint_savings
after_tax_savings = gross_savings * (1 - TAX_RATE)

delta_nwc = np.zeros(PROJECT_LIFE)
incremental_revenue = NUM_AIRCRAFT * revenue_per_ac
nwc = NWC_PCT * incremental_revenue
delta_nwc[0] = nwc[0]
for t in range(1, PROJECT_LIFE):
    delta_nwc[t] = nwc[t] - nwc[t - 1]

salvage_value = NUM_AIRCRAFT * COST_PER_AIRCRAFT * SALVAGE_PCT
book_value_at_end = C0 - depreciation.sum()
salvage_gain = salvage_value - book_value_at_end
salvage_tax  = max(0, salvage_gain * TAX_RATE)
net_salvage  = salvage_value - salvage_tax

fcff = after_tax_savings + tax_shield - fleet_check_capex - delta_nwc
fcff[-1] += net_salvage  # terminal salvage in final year

print("=== FCFF Build-up ($M) — Selected Years ===")
header = (f"{'Yr':>3} {'GrossSav':>10} {'AfterTax':>10} {'TaxShld':>9} "
          f"{'ChkCapEx':>10} {'ΔNWC':>8} {'FCFF':>10}")
print(header)
print("─" * len(header))
for t in [1, 2, 5, 7, 10, 12, 15, 20, 24, 25]:
    i = t - 1
    print(f"{t:3d} {gross_savings[i]:10.1f} {after_tax_savings[i]:10.1f} "
          f"{tax_shield[i]:9.1f} {fleet_check_capex[i]:10.1f} "
          f"{delta_nwc[i]:8.1f} {fcff[i]:10.1f}")
print(f"\nNet salvage value added to Year {PROJECT_LIFE}: ${net_salvage:,.1f}M")
print(f"  (Gross salvage ${salvage_value:,.0f}M − tax on gain ${salvage_tax:,.1f}M)")

---
## 7. Capital Budgeting Metrics: NPV, IRR, MIRR, PI, Break-Even (Issue 6)

| Metric | Formula | Decision Rule |
|---|---|---|
| **NPV** | $-C_0 + \sum \frac{\text{FCFF}_t}{(1+\text{WACC})^t}$ | Accept if > 0 |
| **IRR** | Rate where NPV = 0 | Accept if > WACC |
| **MIRR** | $(FV_{\text{inflows}} / PV_{\text{outflows}})^{1/n} - 1$ | Accept if > WACC |
| **PI** | $PV(\text{future CFs}) / C_0$ | Accept if > 1.0 |

Sources: [CFA Institute](https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2025/capital-investments-and-capital-allocation), [Investopedia MIRR](https://www.investopedia.com/terms/m/mirr.asp)

In [ ]:
# --- NPV ---
discount_factors = (1 + WACC) ** years
pv_fcff = fcff / discount_factors
npv = -C0 + pv_fcff.sum()

# --- IRR ---
cash_flows = np.concatenate([[-C0], fcff])

def npv_at_rate(r):
    t = np.arange(len(cash_flows))
    return np.sum(cash_flows / (1 + r) ** t)

irr = brentq(npv_at_rate, -0.05, 1.0)

# --- MIRR ---
reinvest_rate = WACC
finance_rate  = R_D
positives = np.where(cash_flows > 0, cash_flows, 0)
negatives = np.where(cash_flows < 0, cash_flows, 0)
n = len(cash_flows) - 1
fv_positives = np.sum(positives * (1 + reinvest_rate) ** (n - np.arange(len(cash_flows))))
pv_negatives = np.sum(negatives / (1 + finance_rate) ** np.arange(len(cash_flows)))
mirr = (fv_positives / abs(pv_negatives)) ** (1 / n) - 1

# --- Profitability Index ---
pi = pv_fcff.sum() / C0

# --- Break-even year ---
cumulative_pv = np.cumsum(pv_fcff)
break_even_idx = np.argmax(cumulative_pv >= C0)
break_even_year = years[break_even_idx] if cumulative_pv[break_even_idx] >= C0 else None

# --- Discounted Payback Period (fractional) ---
if break_even_year is not None and break_even_idx > 0:
    shortfall = C0 - cumulative_pv[break_even_idx - 1]
    fraction = shortfall / pv_fcff[break_even_idx]
    discounted_payback = (break_even_year - 1) + fraction
else:
    discounted_payback = break_even_year

print("═" * 50)
print("  CAPITAL BUDGETING DECISION METRICS")
print("═" * 50)
print(f"  NPV                  = ${npv:,.1f}M")
print(f"  IRR                  = {irr:.2%}")
print(f"  MIRR                 = {mirr:.2%}")
print(f"  Profitability Index  = {pi:.4f}")
print(f"  Discounted Payback   = {discounted_payback:.1f} years")
print(f"  WACC (hurdle rate)   = {WACC:.2%}")
print("─" * 50)
accept = npv > 0 and irr > WACC and pi > 1.0
verdict = 'ACCEPT' if accept else 'REJECT'
print(f"  Decision: {verdict}")
print(f"    NPV > 0?          {'✓' if npv > 0 else '✗'}")
print(f"    IRR > WACC?       {'✓' if irr > WACC else '✗'}  ({irr:.2%} vs {WACC:.2%})")
print(f"    PI > 1.0?         {'✓' if pi > 1.0 else '✗'}")
print("═" * 50)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.bar(years, pv_fcff, color='steelblue', alpha=0.7, label='PV of FCFF')
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.set_xlabel('Year')
ax1.set_ylabel('Discounted FCFF ($M)')
ax1.set_title('Annual Discounted Free Cash Flows')
ax1.legend()

ax2.plot(years, cumulative_pv, 'b-o', markersize=4, label='Cumulative PV of FCFF')
ax2.axhline(y=C0, color='r', linestyle='--', linewidth=2, label=f'Investment = ${C0:,.0f}M')
if break_even_year:
    ax2.axvline(x=discounted_payback, color='green', linestyle=':', linewidth=2,
                label=f'Break-even ≈ {discounted_payback:.1f} yr')
ax2.set_xlabel('Year')
ax2.set_ylabel('Cumulative PV ($M)')
ax2.set_title('Cumulative NPV vs Initial Investment')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 8. Financing Scenarios: Purchase vs Lease (Issue 10)

**Scenario A — Debt-Financed Purchase** (60% debt, 40% equity):

$\text{Annual Debt Service} = \text{Debt} \times \frac{R_d}{1 - (1+R_d)^{-n_{\text{loan}}}}$

**Scenario B — Operating Lease** (0.8%/month of aircraft value):

$\text{Annual Lease Cost} = N \times \text{Cost} \times \text{LRF} \times 12$

Sources: [IATA Financing](https://www.iata.org/en/publications/manuals/international-aircraft-financing), [IATA Leases](https://www.iata.org/contentassets/bf8ca67c8bcd4358b3d004b0d6d0916f/ac-leases-4th-edition.pdf)

In [ ]:
LOAN_TERM = 15  # years for aircraft debt
LEASE_RATE_FACTOR = 0.008  # 0.8% per month

# --- Scenario A: Purchase with Debt + Equity ---
debt_amount  = C0 * DEBT_RATIO
equity_amount = C0 * EQUITY_RATIO
annual_debt_service = debt_amount * R_D / (1 - (1 + R_D) ** (-LOAN_TERM))

# Amortization schedule
remaining_principal = debt_amount
interest_payments = np.zeros(PROJECT_LIFE)
principal_payments = np.zeros(PROJECT_LIFE)
for t in range(min(LOAN_TERM, PROJECT_LIFE)):
    interest_payments[t] = remaining_principal * R_D
    principal_payments[t] = annual_debt_service - interest_payments[t]
    remaining_principal -= principal_payments[t]

interest_tax_shield_a = interest_payments * TAX_RATE

# Scenario A FCFF adjustment: add interest tax shield, subtract equity outflow at t=0
fcff_purchase = fcff.copy()
fcff_purchase += interest_tax_shield_a  # interest deductibility benefit

# --- Scenario B: Operating Lease ---
annual_lease_cost = NUM_AIRCRAFT * COST_PER_AIRCRAFT * LEASE_RATE_FACTOR * 12
lease_tax_saving  = annual_lease_cost * TAX_RATE  # lease payments fully deductible
net_lease_cost    = annual_lease_cost - lease_tax_saving

# Under lease, no depreciation shield and no salvage, but no upfront C0
fcff_lease = np.zeros(PROJECT_LIFE)
for t in range(PROJECT_LIFE):
    fcff_lease[t] = (after_tax_savings[t]  # operational savings still accrue
                     - net_lease_cost       # net lease payment replaces ownership cost
                     - fleet_check_capex[t]
                     - delta_nwc[t])

# NPV comparison (lease has no upfront C0)
npv_purchase = -C0 + np.sum(fcff_purchase / discount_factors)
npv_lease    = np.sum(fcff_lease / discount_factors)  # no upfront cost

print("═" * 60)
print("  FINANCING SCENARIO COMPARISON")
print("═" * 60)
print(f"\n  Scenario A: Debt-Financed Purchase")
print(f"    Debt (60%)           = ${debt_amount:,.0f}M")
print(f"    Equity (40%)         = ${equity_amount:,.0f}M")
print(f"    Annual debt service  = ${annual_debt_service:,.1f}M (over {LOAN_TERM} yr)")
print(f"    NPV                  = ${npv_purchase:,.1f}M")
print(f"\n  Scenario B: Operating Lease")
print(f"    Annual lease cost    = ${annual_lease_cost:,.1f}M (gross)")
print(f"    After-tax lease cost = ${net_lease_cost:,.1f}M")
print(f"    NPV                  = ${npv_lease:,.1f}M")
print(f"\n  ► Preferred: {'Purchase' if npv_purchase > npv_lease else 'Lease'} "
      f"(ΔNPV = ${abs(npv_purchase - npv_lease):,.1f}M)")
print("═" * 60)

---
## 9. Sensitivity Analysis — Tornado Diagram (Issue 9a)

One-at-a-time ±20% variation of each key parameter, measuring NPV impact.

Source: [Investopedia — Sensitivity Analysis](https://www.investopedia.com/terms/s/sensitivityanalysis.asp)

In [ ]:
def compute_npv(num_ac, cost_ac, fuel_pct, maint_base, rev_growth,
                rf, beta, eq_prem, rd, debt_ratio, tax, salvage_pct,
                g1, g2, project_life=PROJECT_LIFE):
    """Full NPV recomputation with arbitrary parameters."""
    c0 = num_ac * cost_ac
    re = rf + beta * eq_prem
    wacc = (1 - debt_ratio) * re + debt_ratio * rd * (1 - tax)
    yrs = np.arange(1, project_life + 1)

    macrs = np.array([14.29, 24.49, 17.49, 12.49, 8.93, 8.92, 8.93, 4.46]) / 100
    dep = np.zeros(project_life)
    for i, r in enumerate(macrs):
        if i < project_life:
            dep[i] = c0 * r
    ts = dep * tax

    rev = np.array([ANNUAL_REVENUE_0 * (1 + rev_growth) ** t for t in yrs])
    fs = fuel_pct * rev

    def ms(t):
        if t <= 5:
            return maint_base
        elif t <= 12:
            return maint_base * (1 + g1) ** (t - 5)
        else:
            return maint_base * (1 + g1) ** 7 * (1 + g2) ** (t - 12)

    m_sav = np.array([ms(t) for t in yrs])

    def ck(t):
        cx = 0.0
        if t > 1 and t % C_CHECK_INTERVAL == 0: cx += C_CHECK_COST
        if t > 1 and t % D_CHECK_INTERVAL == 0: cx += D_CHECK_COST
        return cx

    chk = np.array([ck(t) for t in yrs])

    gross = num_ac * (fs + m_sav)
    at = gross * (1 - tax)
    inc_rev = num_ac * rev
    nwc_arr = NWC_PCT * inc_rev
    dnwc = np.zeros(project_life)
    dnwc[0] = nwc_arr[0]
    for i in range(1, project_life):
        dnwc[i] = nwc_arr[i] - nwc_arr[i - 1]

    sv = num_ac * cost_ac * salvage_pct
    bv = c0 - dep.sum()
    sg = sv - bv
    st = max(0, sg * tax)
    ns = sv - st

    cf = at + ts - num_ac * chk - dnwc
    cf[-1] += ns

    df = (1 + wacc) ** yrs
    return -c0 + np.sum(cf / df)

base_npv = compute_npv(NUM_AIRCRAFT, COST_PER_AIRCRAFT, FUEL_SAVINGS_PCT,
                       MAINT_SAVINGS_BASE, REVENUE_GROWTH, R_F, BETA,
                       EQUITY_PREMIUM, R_D, DEBT_RATIO, TAX_RATE,
                       SALVAGE_PCT, G1_MID, G2_LATE)

params = [
    ('Aircraft cost ($M)',     'cost_ac',    COST_PER_AIRCRAFT),
    ('Fuel savings %',        'fuel_pct',   FUEL_SAVINGS_PCT),
    ('Beta (β)',              'beta',       BETA),
    ('Cost of debt (Rd)',     'rd',         R_D),
    ('Revenue growth',        'rev_growth', REVENUE_GROWTH),
    ('Maint. savings base',   'maint_base', MAINT_SAVINGS_BASE),
    ('Salvage %',            'salvage_pct',SALVAGE_PCT),
    ('Tax rate',             'tax',        TAX_RATE),
    ('Debt ratio (D/V)',     'debt_ratio', DEBT_RATIO),
    ('Late-life maint. esc.','g2',         G2_LATE),
]

base_kwargs = dict(num_ac=NUM_AIRCRAFT, cost_ac=COST_PER_AIRCRAFT,
                   fuel_pct=FUEL_SAVINGS_PCT, maint_base=MAINT_SAVINGS_BASE,
                   rev_growth=REVENUE_GROWTH, rf=R_F, beta=BETA,
                   eq_prem=EQUITY_PREMIUM, rd=R_D, debt_ratio=DEBT_RATIO,
                   tax=TAX_RATE, salvage_pct=SALVAGE_PCT, g1=G1_MID, g2=G2_LATE)

swing = 0.20
results = []
for label, key, base_val in params:
    kw_low = base_kwargs.copy()
    kw_high = base_kwargs.copy()
    kw_low[key]  = base_val * (1 - swing)
    kw_high[key] = base_val * (1 + swing)
    npv_low  = compute_npv(**kw_low)
    npv_high = compute_npv(**kw_high)
    results.append((label, npv_low, npv_high))

results.sort(key=lambda x: abs(x[2] - x[1]))

fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(results))
for i, (label, lo, hi) in enumerate(results):
    left  = min(lo, hi) - base_npv
    right = max(lo, hi) - base_npv
    ax.barh(i, right - left, left=left + base_npv, height=0.6,
            color='steelblue' if hi > lo else 'salmon', alpha=0.8)
    ax.barh(i, -(right - left), left=left + base_npv, height=0.6,
            color='salmon' if hi > lo else 'steelblue', alpha=0.0)
    lo_delta = lo - base_npv
    hi_delta = hi - base_npv
    ax.barh(i, lo_delta, left=base_npv, height=0.6,
            color='salmon', alpha=0.7)
    ax.barh(i, hi_delta, left=base_npv, height=0.6,
            color='steelblue', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels([r[0] for r in results])
ax.axvline(x=base_npv, color='black', linewidth=1.5, linestyle='-')
ax.set_xlabel('NPV ($M)')
ax.set_title(f'Tornado Diagram: NPV Sensitivity to ±{swing:.0%} Parameter Changes\n'
             f'(Base NPV = ${base_npv:,.0f}M)')
ax.legend(['−20%', '+20%'], loc='lower right')
plt.tight_layout()
plt.show()

print(f"\n{'Parameter':<25} {'−20% NPV':>12} {'Base NPV':>12} {'+20% NPV':>12} {'Swing':>12}")
print("─" * 75)
for label, lo, hi in reversed(results):
    print(f"{label:<25} ${lo:>10,.0f}M ${base_npv:>10,.0f}M ${hi:>10,.0f}M ${abs(hi-lo):>10,.0f}M")

---
## 10. Monte Carlo Simulation (Issue 9b)

10,000 iterations sampling from probability distributions for key uncertain parameters.

$$\text{NPV}_i = -C_0 + \sum_{t=1}^{n} \frac{\text{FCFF}_t(\omega_i)}{(1+\text{WACC}(\omega_i))^t}$$

Source: [CFA Institute — Capital Investments](https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2025/capital-investments-and-capital-allocation)

In [ ]:
N_SIM = 10_000

mc_npvs = np.zeros(N_SIM)
mc_irrs = np.zeros(N_SIM)

for i in range(N_SIM):
    sim_cost     = np.random.triangular(96, 120, 150)
    sim_fuel_pct = np.random.normal(0.20, 0.03)
    sim_maint    = np.random.normal(2.0, 0.4)
    sim_rev_g    = np.random.normal(0.03, 0.01)
    sim_beta     = np.random.normal(1.30, 0.20)
    sim_rf       = np.random.normal(0.0425, 0.005)
    sim_rd       = np.random.normal(0.045, 0.005)
    sim_salvage  = np.random.uniform(0.08, 0.25)
    sim_g2       = np.random.uniform(0.06, 0.12)

    sim_fuel_pct = np.clip(sim_fuel_pct, 0.05, 0.35)
    sim_maint    = np.clip(sim_maint, 0.5, 4.0)
    sim_beta     = np.clip(sim_beta, 0.5, 2.5)

    mc_npvs[i] = compute_npv(
        num_ac=NUM_AIRCRAFT, cost_ac=sim_cost, fuel_pct=sim_fuel_pct,
        maint_base=sim_maint, rev_growth=sim_rev_g, rf=sim_rf,
        beta=sim_beta, eq_prem=EQUITY_PREMIUM, rd=sim_rd,
        debt_ratio=DEBT_RATIO, tax=TAX_RATE, salvage_pct=sim_salvage,
        g1=G1_MID, g2=sim_g2
    )

prob_loss = np.mean(mc_npvs < 0) * 100

print("═" * 55)
print("  MONTE CARLO SIMULATION RESULTS  (n = {:,})".format(N_SIM))
print("═" * 55)
print(f"  Mean NPV             = ${np.mean(mc_npvs):,.0f}M")
print(f"  Median NPV           = ${np.median(mc_npvs):,.0f}M")
print(f"  Std Dev              = ${np.std(mc_npvs):,.0f}M")
print(f"  5th percentile       = ${np.percentile(mc_npvs, 5):,.0f}M")
print(f"  95th percentile      = ${np.percentile(mc_npvs, 95):,.0f}M")
print(f"  P(NPV < 0)           = {prob_loss:.1f}%")
print(f"  P(NPV > 0)           = {100 - prob_loss:.1f}%")
print("═" * 55)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.hist(mc_npvs, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(x=0, color='red', linewidth=2, linestyle='--', label='NPV = 0')
ax1.axvline(x=np.mean(mc_npvs), color='green', linewidth=2, label=f'Mean = ${np.mean(mc_npvs):,.0f}M')
ax1.axvline(x=np.percentile(mc_npvs, 5), color='orange', linewidth=1.5, linestyle=':',
            label=f'5th pctl = ${np.percentile(mc_npvs, 5):,.0f}M')
ax1.axvline(x=np.percentile(mc_npvs, 95), color='orange', linewidth=1.5, linestyle=':',
            label=f'95th pctl = ${np.percentile(mc_npvs, 95):,.0f}M')
ax1.set_xlabel('NPV ($M)')
ax1.set_ylabel('Frequency')
ax1.set_title(f'Monte Carlo NPV Distribution (n={N_SIM:,})')
ax1.legend(fontsize=9)

sorted_npv = np.sort(mc_npvs)
cdf = np.arange(1, N_SIM + 1) / N_SIM
ax2.plot(sorted_npv, cdf, 'b-', linewidth=1.5)
ax2.axvline(x=0, color='red', linewidth=2, linestyle='--')
ax2.axhline(y=prob_loss / 100, color='red', linewidth=1, linestyle=':', alpha=0.7)
ax2.set_xlabel('NPV ($M)')
ax2.set_ylabel('Cumulative Probability')
ax2.set_title(f'CDF of NPV — P(loss) = {prob_loss:.1f}%')
ax2.fill_betweenx(cdf, sorted_npv, 0, where=(sorted_npv < 0),
                  color='red', alpha=0.15, label=f'Loss region ({prob_loss:.1f}%)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 11. Comparison: Naive Model vs Improved Model

In [ ]:
# --- Reproduce naive model ---
r_naive = 0.08
c1_naive = 0.20 * 15.0
c2_naive = 2.0
g_naive = 0.05

naive_cumulative = 0.0
naive_break_even = None
naive_annual_pv = np.zeros(PROJECT_LIFE)
for t in range(1, PROJECT_LIFE + 1):
    ct = NUM_AIRCRAFT * (c1_naive + c2_naive * (1 + g_naive) ** (t - 1))
    pv = ct / (1 + r_naive) ** t
    naive_annual_pv[t - 1] = pv
    naive_cumulative += pv
    if naive_cumulative >= C0 and naive_break_even is None:
        naive_break_even = t

naive_npv = -C0 + naive_cumulative

print("╔" + "═" * 62 + "╗")
print("║" + "  NAIVE vs IMPROVED MODEL — SIDE-BY-SIDE COMPARISON".center(62) + "║")
print("╠" + "═" * 62 + "╣")
print(f"║  {'Metric':<30} {'Naive':>12} {'Improved':>14}  ║")
print("║  " + "─" * 58 + "  ║")
print(f"║  {'Discount rate':<30} {'8.00%':>12} {WACC:>13.2%}   ║")
print(f"║  {'Discount rate source':<30} {'Assumed':>12} {'CAPM → WACC':>14}  ║")
print(f"║  {'Depreciation tax shield':<30} {'None':>12} {'MACRS 7-yr':>14}  ║")
print(f"║  {'Tax effects on savings':<30} {'None':>12} {'21% corp tax':>14}  ║")
print(f"║  {'Salvage value':<30} {'$0M':>12} {f'${net_salvage:,.0f}M':>14}  ║")
print(f"║  {'Maintenance model':<30} {'5% flat':>12} {'Bathtub+Chk':>14}  ║")
print(f"║  {'Revenue growth':<30} {'0%':>12} {'3%/yr':>14}  ║")
print(f"║  {'Financing structure':<30} {'Lump sum':>12} {'D/E modeled':>14}  ║")
print(f"║  {'Sensitivity analysis':<30} {'None':>12} {'Tornado+MC':>14}  ║")
print("║  " + "─" * 58 + "  ║")
n_be_str = f"{naive_break_even} years" if naive_break_even else ">25 years"
i_be_str = f"{discounted_payback:.1f} years" if break_even_year else ">25 years"
print(f"║  {'NPV @ 25 years':<30} {f'${naive_npv:,.0f}M':>12} {f'${npv:,.0f}M':>14}  ║")
print(f"║  {'Break-even':<30} {n_be_str:>12} {i_be_str:>14}  ║")
print(f"║  {'IRR':<30} {'N/A':>12} {f'{irr:.2%}':>14}  ║")
print(f"║  {'MIRR':<30} {'N/A':>12} {f'{mirr:.2%}':>14}  ║")
print(f"║  {'Profitability Index':<30} {'N/A':>12} {f'{pi:.4f}':>14}  ║")
print(f"║  {'P(NPV < 0) Monte Carlo':<30} {'N/A':>12} {f'{prob_loss:.1f}%':>14}  ║")
print("╚" + "═" * 62 + "╝")

# --- Visual comparison ---
fig, ax = plt.subplots(figsize=(12, 6))
naive_cum = np.cumsum(naive_annual_pv)
improved_cum = cumulative_pv

ax.plot(years, naive_cum, 'r--s', markersize=4, linewidth=2, label='Naive model (flat 8%, no tax/depr)')
ax.plot(years, improved_cum, 'b-o', markersize=4, linewidth=2, label='Improved model (WACC, FCFF, MACRS)')
ax.axhline(y=C0, color='k', linestyle=':', linewidth=2, label=f'Investment = ${C0:,.0f}M')
ax.fill_between(years, naive_cum, improved_cum, alpha=0.1, color='green',
                label='Improvement from corrections')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative PV of Cash Flows ($M)')
ax.set_title('Naive vs Improved Break-Even Analysis')
ax.legend()
plt.tight_layout()
plt.show()

---
## References

| # | Source | URL |
|---|---|---|
| 1 | Damodaran — Cost of Capital | [PDF](https://pages.stern.nyu.edu/~adamodar/pdfiles/papers/costofcapital.pdf) |
| 2 | Investopedia — CAPM | [Link](https://www.investopedia.com/terms/c/capm.asp) |
| 3 | NBAA — Aircraft Depreciation | [Link](https://nbaa.org/flight-department-administration/tax-issues/depreciation/depreciation-schedule-for-business-aircraft/) |
| 4 | IRS Publication 946 | [PDF](https://www.irs.gov/pub/irs-pdf/p946.pdf) |
| 5 | Investopedia — MACRS | [Link](https://www.investopedia.com/terms/m/macrs.asp) |
| 6 | IATA — Aircraft Acquisition Disclosure | [PDF](https://www.iata.org/contentassets/4a4b100c43794398baf73dcea6b5ad42/airline-disclosure-guide-aircraft-acquisition.pdf) |
| 7 | Investopedia — FCFF | [Link](https://www.investopedia.com/terms/f/freecashflowfirm.asp) |
| 8 | CFI — FCFF Formula | [Link](https://corporatefinanceinstitute.com/resources/financial-modeling/free-cash-flow-to-firm-fcff) |
| 9 | CFA Institute — Capital Investments | [Link](https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2025/capital-investments-and-capital-allocation) |
| 10 | Investopedia — MIRR | [Link](https://www.investopedia.com/terms/m/mirr.asp) |
| 11 | IATA — Aging Aircraft Costs | [PDF](https://www.iata.org/contentassets/bf8ca67c8bcd4358b3d004b0d6d0916f/mcaa-1sted-2018.pdf) |
| 12 | IATA — Economic Reports | [Link](https://www.iata.org/en/iata-repository/publications/economic-reports/) |
| 13 | Investopedia — Sensitivity Analysis | [Link](https://www.investopedia.com/terms/s/sensitivityanalysis.asp) |
| 14 | IATA — Aircraft Financing | [Link](https://www.iata.org/en/publications/manuals/international-aircraft-financing) |
| 15 | IATA — Aircraft Leases | [PDF](https://www.iata.org/contentassets/bf8ca67c8bcd4358b3d004b0d6d0916f/ac-leases-4th-edition.pdf) |
| 16 | CFI — WACC Calculator | [Link](https://corporatefinanceinstitute.com/resources/financial-modeling/wacc-calculator) |